# 🔥 Full RAG + Agentic RAG Classroom Lab
### *Built with LlamaIndex · ChromaDB · Ollama · OpenRouter · Sentence-Transformers*

---

**Trainer:** Senior RAG Engineer  
**Level:** Intermediate → Advanced  
**Duration:** 3–4 hours (full lab)  
**Goal:** Build a production-grade RAG system from scratch using only free, open-source tools.

---

## 📚 Section 1 — Lab Introduction

### What is RAG?
**Retrieval-Augmented Generation (RAG)** is an AI architecture that combines:
- 🔍 **Retrieval** — Finding relevant documents from a knowledge base
- 🧠 **Augmentation** — Injecting retrieved context into the prompt
- ✍️ **Generation** — LLM produces grounded, factual answers

### Why Do We Need RAG?
| Problem | RAG Solution |
|---|---|
| LLM hallucinations | Grounds answers in real documents |
| Knowledge cutoff | Uses your latest documents |
| Private data | Works with local files |
| Long documents | Retrieves only relevant chunks |
| Cost | Avoids full-document prompting |

### 🏗️ Architecture Diagram

```
┌─────────────────────────────────────────────────────────────────┐
│                    RAG PIPELINE ARCHITECTURE                    │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  INDEXING PHASE (offline)                                       │
│  ┌──────────┐   ┌──────────┐   ┌──────────┐   ┌────────────┐  │
│  │  PDFs /  │──▶│  Chunk   │──▶│  Embed   │──▶│  ChromaDB  │  │
│  │  Docs    │   │  Text    │   │  Model   │   │  Vector DB │  │
│  └──────────┘   └──────────┘   └──────────┘   └────────────┘  │
│                                                                 │
│  QUERY PHASE (online)                                           │
│  ┌──────────┐   ┌──────────┐   ┌──────────┐   ┌────────────┐  │
│  │  User    │──▶│  Embed   │──▶│ Retrieve │──▶│  LLM       │  │
│  │  Query   │   │  Query   │   │  Top-K   │   │  Generate  │  │
│  └──────────┘   └──────────┘   └──────────┘   └────────────┘  │
│                                                        │        │
│                                               ┌────────▼─────┐ │
│                                               │   Answer +   │ │
│                                               │  Citations   │ │
│                                               └──────────────┘ │
│                                                                 │
│  AGENTIC RAG (advanced)                                         │
│  ┌──────────┐   ┌──────────┐   ┌──────────┐   ┌────────────┐  │
│  │  User    │──▶│  Agent   │──▶│ Plan /   │──▶│  Tools:    │  │
│  │  Query   │   │  (LLM)   │   │ Reason   │   │ RAG/Search │  │
│  └──────────┘   └──────────┘   └──────────┘   └────────────┘  │
│                      ▲                                │         │
│                      └────────── Feedback ────────────┘         │
└─────────────────────────────────────────────────────────────────┘
```

### 🌐 Local vs Cloud RAG
| Feature | Local RAG | Cloud RAG |
|---|---|---|
| Privacy | ✅ Full | ❌ Data sent to cloud |
| Cost | ✅ Free | ❌ Pay-per-token |
| Speed | 🔶 GPU dependent | ✅ Fast |
| Quality | 🔶 Model dependent | ✅ State-of-art |
| Offline | ✅ Yes | ❌ No |

### 🛠️ Tools Used in This Lab
| Tool | Role | Cost |
|---|---|---|
| **LlamaIndex** | RAG framework | Free |
| **ChromaDB** | Vector database | Free |
| **Ollama** | Local LLM runner | Free |
| **OpenRouter** | Cloud LLM API | Free tier |
| **nomic-embed-text** | Embedding model | Free (Ollama) |
| **sentence-transformers** | Fallback embeddings | Free |
| **pypdf** | PDF loading | Free |

---
## ⚙️ Section 2 — Environment Setup

> **💡 Trainer Note:** Run these cells once at the start. If you are in Google Colab, all installs will reset on reconnect.

In [ ]:
# ============================================================
# CELL 2.1 — Install all required packages
# ============================================================
# Run this cell first. Takes ~2 minutes on first run.

!pip install -q llama-index llama-index-core
!pip install -q llama-index-llms-openai-like
!pip install -q llama-index-llms-ollama
!pip install -q llama-index-embeddings-ollama
!pip install -q llama-index-embeddings-huggingface
!pip install -q llama-index-vector-stores-chroma
!pip install -q llama-index-agent-openai
!pip install -q chromadb
!pip install -q pypdf
!pip install -q sentence-transformers
!pip install -q openai
!pip install -q requests
!pip install -q python-dotenv

print("✅ All packages installed successfully!")

In [1]:
# ============================================================
# CELL 2.2 — Core imports
# ============================================================

import os
import sys
import json
import warnings
import textwrap
from pathlib import Path

warnings.filterwarnings("ignore")

# LlamaIndex core
from llama_index.core import (
    VectorStoreIndex,
    SimpleDirectoryReader,
    StorageContext,
    Settings,
    Document,
)
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.retrievers import VectorIndexRetriever
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.response_synthesizers import get_response_synthesizer
from llama_index.core.tools import QueryEngineTool, ToolMetadata
from llama_index.core.agent import ReActAgent

# Vector Store
import chromadb
from llama_index.vector_stores.chroma import ChromaVectorStore

print("✅ All imports successful!")
print(f"Python version: {sys.version}")

✅ All imports successful!
Python version: 3.13.13 (tags/v3.13.13:01104ce, Apr  7 2026, 19:25:48) [MSC v.1944 64 bit (AMD64)]


---
## 🤖 Section 3 — Load LLM (Ollama OR OpenRouter)

> **Choose ONE option below. Comment out the other.**
>
> - **Option A** = Local Ollama (fully free, private, runs on your machine)
> - **Option B** = OpenRouter (cloud, free tier available, needs API key)

In [2]:
# ============================================================
# CELL 3.0 — Check if Ollama is running
# ============================================================
import subprocess
import requests as req

def check_ollama():
    try:
        r = req.get("http://localhost:11434/api/tags", timeout=3)
        if r.status_code == 200:
            models = [m['name'] for m in r.json().get('models', [])]
            print("✅ Ollama is running!")
            print(f"📦 Available models: {models if models else 'No models pulled yet'}")
            return True, models
    except Exception:
        pass
    print("❌ Ollama not detected. Start with: 'ollama serve' in terminal")
    print("📥 Then pull models: 'ollama pull qwen2.5:7b' and 'ollama pull nomic-embed-text'")
    return False, []

ollama_running, available_models = check_ollama()

✅ Ollama is running!
📦 Available models: ['nomic-embed-text:latest', 'phi4-mini:latest', 'qwen3:1.7b']


In [ ]:
# ============================================================
# CELL 3.1 — OPTION A: Local LLM via Ollama
# ============================================================
# Prerequisites:
#   1. Install Ollama: https://ollama.ai
#   2. Run: ollama pull qwen2.5:7b
#   3. Run: ollama serve  (in separate terminal)

from llama_index.llms.ollama import Ollama

# ----- CONFIG -----
OLLAMA_MODEL = "qwen2.5:7b"   # or "llama3.1:8b", "mistral:7b"
OLLAMA_BASE_URL = "http://localhost:11434"
TEMPERATURE = 0.1              # Lower = more factual
REQUEST_TIMEOUT = 120.0        # Seconds (increase for slow machines)
# ------------------

llm_ollama = Ollama(
    model=OLLAMA_MODEL,
    base_url=OLLAMA_BASE_URL,
    temperature=TEMPERATURE,
    request_timeout=REQUEST_TIMEOUT,
)

# Quick test
if ollama_running:
    try:
        test_resp = llm_ollama.complete("Say 'RAG system ready!' in exactly 5 words.")
        print(f"✅ Ollama LLM test: {test_resp}")
        USE_OLLAMA = True
    except Exception as e:
        print(f"⚠️  Ollama model error: {e}")
        print(f"Try: ollama pull {OLLAMA_MODEL}")
        USE_OLLAMA = False
else:
    print("⏭️  Skipping Ollama test — not running. Use Option B below.")
    USE_OLLAMA = False

In [4]:
# ============================================================
# CELL 3.2 — OPTION B: OpenRouter (Cloud LLM)
# ============================================================
# Prerequisites:
#   1. Sign up at https://openrouter.ai (free)
#   2. Get API key from dashboard
#   3. Set OPENROUTER_API_KEY below OR in environment variable

from llama_index.llms.openai_like import OpenAILike

# ----- CONFIG -----
# Option 1: Hardcode (not recommended for production)
OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY")

# Option 2: Load from .env file
# from dotenv import load_dotenv
# load_dotenv()  # Create .env file with: OPENROUTER_API_KEY=sk-or-v1-...
# OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY")

# Available free models on OpenRouter:
# "qwen/qwen-2.5-7b-instruct"  (recommended)
# "mistralai/mistral-7b-instruct"
# "google/gemma-2-9b-it:free"
# "meta-llama/llama-3.1-8b-instruct:free"
OPENROUTER_MODEL = "qwen/qwen-2.5-7b-instruct"
TEMPERATURE = 0.1
# ------------------

llm_openrouter = OpenAILike(
    model=OPENROUTER_MODEL,
    api_base="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
    temperature=TEMPERATURE,
    max_tokens=2048,
    is_chat_model=True,
    default_headers={
        "HTTP-Referer": "https://classroom.rag.lab",
        "X-Title": "RAG Classroom Lab",
    }
)

# Test OpenRouter connection
if OPENROUTER_API_KEY and not OPENROUTER_API_KEY.endswith("YOUR_KEY_HERE"):
    try:
        test_resp = llm_openrouter.complete("Say 'OpenRouter ready!' in exactly 3 words.")
        print(f"✅ OpenRouter LLM test: {test_resp}")
        USE_OPENROUTER = True
    except Exception as e:
        print(f"⚠️  OpenRouter error: {e}")
        USE_OPENROUTER = False
else:
    print("⏭️  OpenRouter key not set. Using Ollama (Option A).")
    USE_OPENROUTER = False

✅ OpenRouter LLM test: OpenRouter ready!


In [5]:
# ============================================================
# CELL 3.3 — Select Active LLM
# ============================================================
# Automatically picks the available LLM

if USE_OPENROUTER:
    llm = llm_openrouter
    print(f"🌐 Using OpenRouter: {OPENROUTER_MODEL}")
elif USE_OLLAMA:
    llm = llm_ollama
    print(f"🏠 Using Ollama: {OLLAMA_MODEL}")
else:
    print("⚠️  No LLM available. Set up Ollama or OpenRouter first!")
    # Fallback mock LLM for notebook exploration
    class MockLLM:
        def complete(self, prompt):
            return type('R', (), {'text': '[MOCK] LLM response - configure Ollama or OpenRouter'})() 
    llm = MockLLM()

# Apply globally to LlamaIndex
Settings.llm = llm
print("✅ LLM configured in LlamaIndex Settings")

🌐 Using OpenRouter: qwen/qwen-2.5-7b-instruct
✅ LLM configured in LlamaIndex Settings


---
## 📄 Section 4 — Load Sample PDFs

> We'll create 3 sample PDFs programmatically using public domain content.
> In a real project, replace these with your own PDF files.

> **💡 Trainer Note:** The chunking strategy is critical. Too small = lost context. Too large = irrelevant noise.

In [31]:
# ============================================================
# CELL 4.1 — Create Sample PDFs (Public Domain Content)
# ============================================================
# We use pypdf and reportlab to create test documents.
# Install reportlab if not present:

!pip install -q reportlab

from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib.units import inch

# Create sample_docs directory
os.makedirs("sample_docs", exist_ok=True)

def create_pdf(filename, title, sections):
    """Helper to create a sample PDF with multiple sections."""
    doc = SimpleDocTemplate(filename, pagesize=letter)
    styles = getSampleStyleSheet()
    story = []
    
    # Title
    story.append(Paragraph(title, styles['Title']))
    story.append(Spacer(1, 0.3*inch))
    
    for sec_title, content in sections.items():
        story.append(Paragraph(sec_title, styles['Heading1']))
        story.append(Spacer(1, 0.1*inch))
        for para in content:
            story.append(Paragraph(para, styles['Normal']))
            story.append(Spacer(1, 0.1*inch))
    
    doc.build(story)
    print(f"📄 Created: {filename}")

# ---- PDF 1: AI Research Report ----
create_pdf(
    "sample_docs/ai_research_report.pdf",
    "Artificial Intelligence Research Report 2024",
    {
        "Section 1: Executive Summary": [
            "This report examines the state of artificial intelligence research in 2024, with a focus on large language models, multimodal systems, and real-world deployment challenges.",
            "Key findings include: (1) LLM capabilities have expanded dramatically, (2) alignment research remains an open challenge, (3) deployment costs have decreased by 60% year-over-year.",
            "The global AI market reached $500 billion in 2024, with enterprise adoption growing at 35% annually."
        ],
        "Section 2: Large Language Models": [
            "Large language models (LLMs) have demonstrated emergent capabilities including chain-of-thought reasoning, tool use, and multi-step problem solving.",
            "Key milestones in 2024: GPT-5 was released in March 2024 with 1 trillion parameters. Claude 3 achieved human-level performance on 87% of standardized benchmarks.",
            "Mixture-of-Experts (MoE) architectures became the dominant paradigm, reducing inference costs while maintaining performance. Models like Mistral and Qwen adopted this approach.",
            "Training data quality emerged as more important than quantity. Curated datasets of 10 billion tokens outperformed raw 100 billion token corpora on downstream tasks."
        ],
        "Section 3: RAG and Knowledge Systems": [
            "Retrieval-Augmented Generation (RAG) became the standard architecture for enterprise AI in 2024. Over 70% of production AI systems used some form of RAG.",
            "Key RAG innovations: (1) Hybrid search combining dense and sparse retrieval, (2) Agentic RAG with multi-hop reasoning, (3) Graph RAG for complex relationships.",
            "Vector databases saw explosive growth. ChromaDB reported 500% growth in open-source deployments. Qdrant and Weaviate secured major enterprise contracts.",
            "Embedding model quality became a key differentiator. nomic-embed-text and E5 models offered competitive performance to proprietary alternatives."
        ],
        "Section 4: Key Dates and Events": [
            "January 2024: OpenAI released the GPT-4o model with native multimodal capabilities.",
            "March 2024: Anthropic published research on Constitutional AI v2, improving alignment methods.",
            "June 2024: The EU AI Act came into effect, requiring risk assessments for high-risk AI systems.",
            "September 2024: Meta released LLaMA 3.1 with a 405B parameter variant under open license.",
            "December 2024: The first international AI Safety Summit produced the Seoul Declaration."
        ]
    }
)

# ---- PDF 2: Climate Change Policy ----
create_pdf(
    "sample_docs/climate_policy_brief.pdf",
    "Climate Change Policy Brief: 2024 Assessment",
    {
        "Section 1: Overview": [
            "Global average temperatures in 2024 exceeded pre-industrial levels by 1.5°C for the first time in recorded history, crossing the critical threshold set by the Paris Agreement.",
            "This policy brief examines the implications of this milestone and reviews national responses from the G20 nations.",
            "The report was compiled using data from NOAA, NASA, and the Copernicus Climate Change Service."
        ],
        "Section 2: Carbon Emissions Comparison": [
            "Topic A - Developed Nations: The United States reduced carbon emissions by 12% in 2024 through renewable energy expansion and coal phase-outs. The EU achieved 15% reduction through aggressive carbon pricing.",
            "Topic B - Developing Nations: China's emissions plateaued in 2024 as solar capacity additions exceeded 300 GW. India accelerated its renewable transition but remained heavily coal-dependent for baseload power.",
            "Comparison: Developed nations collectively reduced emissions by 8%, while developing nations saw a net increase of 3% due to economic growth. Per-capita emissions remain 3x higher in developed nations."
        ],
        "Section 3: Renewable Energy Progress": [
            "Solar power achieved grid parity in 130 countries by 2024. The levelized cost of solar fell below $0.02 per kWh in sun-rich regions.",
            "Wind energy capacity grew by 25% globally. Offshore wind emerged as the dominant new installation type in Europe and parts of Asia.",
            "Battery storage costs fell 40% year-over-year, making 24/7 renewable power increasingly viable for industrial applications."
        ],
        "Section 4: Policy Recommendations": [
            "Recommendation 1: Implement carbon border adjustment mechanisms to prevent carbon leakage.",
            "Recommendation 2: Increase climate finance for developing nations to $500 billion annually by 2030.",
            "Recommendation 3: Phase out fossil fuel subsidies, which totaled $7 trillion globally in 2023.",
            "Recommendation 4: Accelerate just transition programs for fossil fuel workers and communities."
        ]
    }
)

# ---- PDF 3: Machine Learning Technical Guide ----
create_pdf(
    "sample_docs/ml_technical_guide.pdf",
    "Practical Machine Learning: A Technical Guide",
    {
        "Section 1: Introduction to ML Systems": [
            "Machine learning systems consist of three core components: data pipelines, model training infrastructure, and inference serving systems.",
            "A well-designed ML system should be reproducible, scalable, and monitorable. These properties are often called the ML system design triangle.",
            "The field of MLOps emerged to bridge the gap between data science research and production engineering, drawing on DevOps principles."
        ],
        "Section 2: Model Training Best Practices": [
            "Data quality trumps data quantity. A clean dataset of 100,000 examples typically outperforms a noisy dataset of 10 million examples.",
            "Use stratified sampling for imbalanced datasets. Techniques like SMOTE and class-weighted loss functions can address class imbalance.",
            "Always establish a baseline model before applying complex architectures. A logistic regression baseline reveals the signal strength in your data.",
            "Hyperparameter tuning with Bayesian optimization typically outperforms grid search by 10-30% while requiring fewer compute cycles."
        ],
        "Section 3: Evaluation Metrics": [
            "Classification metrics: Accuracy is misleading for imbalanced datasets. Use F1-score, AUC-ROC, and precision-recall curves instead.",
            "Regression metrics: RMSE penalizes large errors heavily. MAE is more robust to outliers. Choose based on your business requirements.",
            "RAG evaluation: Use RAGAS framework with metrics: faithfulness (is the answer grounded?), answer relevancy, context precision, and context recall."
        ],
        "Section 4: Deployment Strategies": [
            "Blue-green deployment allows zero-downtime model updates by running two production environments simultaneously.",
            "Shadow mode testing runs new models in parallel with production models, comparing outputs before full cutover.",
            "Canary releases gradually shift traffic to new models, allowing monitoring of real-world performance before full deployment."
        ]
    }
)

print("\n✅ All sample PDFs created in ./sample_docs/")
print("Files:", os.listdir("sample_docs"))

📄 Created: sample_docs/ai_research_report.pdf
📄 Created: sample_docs/climate_policy_brief.pdf
📄 Created: sample_docs/ml_technical_guide.pdf

✅ All sample PDFs created in ./sample_docs/
Files: ['ai_research_report.pdf', 'climate_policy_brief.pdf', 'linkedin.pdf', 'ml_technical_guide.pdf']


In [ ]:
# ============================================================
# CELL 4.2 — Load PDFs, Extract Text, Chunk, Show Metadata
# ============================================================

# Load all PDFs from directory
print("📖 Loading PDF documents...")
reader = SimpleDirectoryReader(
    input_dir="./sample_docs",
    required_exts=[".pdf"],
    recursive=False,
)
documents = reader.load_data()
print(f"✅ Loaded {len(documents)} document pages")

# Show sample metadata
print("\n📋 Sample document metadata:")
for i, doc in enumerate(documents[:3]):
    print(f"  Doc {i}: file={doc.metadata.get('file_name', 'N/A')}, "
          f"page={doc.metadata.get('page_label', 'N/A')}, "
          f"chars={len(doc.text)}")

print("\n📝 Sample text from first chunk:")
print("-" * 60)
print(documents[0].text[:500])
print("-" * 60)

In [ ]:
%pip install docling 
%pip install accelerate
%pip install torchvision
%pip install transformers
%pip install sentencepiece

In [6]:
# ============================================================
# CELL 4.2 — Load PDFs Using Docling (Recommended)
# ============================================================

from pathlib import Path
from docling.document_converter import DocumentConverter
from llama_index.core import Document

print("📖 Loading PDFs with Docling...")

pdf_dir = Path("./sample_docs")

converter = DocumentConverter()

documents = []

pdf_files = list(pdf_dir.glob("*.pdf"))

print(f"📂 Found {len(pdf_files)} PDF files")

for pdf_file in pdf_files:

    try:
        print(f"\n🔍 Processing: {pdf_file.name}")

        result = converter.convert(str(pdf_file))

        text = result.document.export_to_markdown()

        if len(text.strip()) < 50:
            print("⚠️ Very little text extracted")
            continue

        documents.append(
            Document(
                text=text,
                metadata={
                    "file_name": pdf_file.name
                }
            )
        )

        print(f"✅ Extracted {len(text):,} characters")

    except Exception as ex:
        print(f"❌ Failed: {pdf_file.name}")
        print(ex)

print("\n" + "=" * 60)
print(f"📚 Total Documents Loaded: {len(documents)}")
print("=" * 60)

if documents:

    sample = documents[0].text[:2000]

    print("\n📄 Sample Extracted Content")
    print("-" * 60)
    print(sample)
    print("-" * 60)

    print("\n📋 Metadata")
    print(documents[0].metadata)

📖 Loading PDFs with Docling...
📂 Found 3 PDF files

🔍 Processing: ai_research_report.pdf


[INFO] 2026-06-09 02:01:24,418 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-06-09 02:01:24,492 [RapidOCR] download_file.py:60: File exists and is valid: D:\Trainings\AI\abu dhabhi agent\.venv\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_mobile.onnx
[INFO] 2026-06-09 02:01:24,494 [RapidOCR] main.py:57: Using D:\Trainings\AI\abu dhabhi agent\.venv\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_mobile.onnx
[INFO] 2026-06-09 02:01:24,715 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-06-09 02:01:24,719 [RapidOCR] download_file.py:60: File exists and is valid: D:\Trainings\AI\abu dhabhi agent\.venv\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-06-09 02:01:24,721 [RapidOCR] main.py:57: Using D:\Trainings\AI\abu dhabhi agent\.venv\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-06-09 02:01:24,853 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-06-09 02:01:2

✅ Extracted 2,382 characters

🔍 Processing: climate_policy_brief.pdf
✅ Extracted 1,982 characters

🔍 Processing: ml_technical_guide.pdf
✅ Extracted 1,935 characters

📚 Total Documents Loaded: 3

📄 Sample Extracted Content
------------------------------------------------------------
## Artificial Intelligence Research Report 2024

## Section 1: Executive Summary

This report examines the state of artificial intelligence research in 2024, with a focus on large language models, multimodal systems, and real-world deployment challenges.

Key findings include: (1) LLM capabilities have expanded dramatically, (2) alignment research remains an open challenge, (3) deployment costs have decreased by 60% year-over-year.

The global AI market reached $500 billion in 2024, with enterprise adoption growing at 35% annually.

## Section 2: Large Language Models

Large language models (LLMs) have demonstrated emergent capabilities including chain-of-thought reasoning, tool use, and multi-step problem s

In [ ]:
%pip uninstall fitz -y
%pip install pymupdf

In [ ]:
import fitz

pdf = fitz.open("./sample_docs/ai_research_report.pdf")

print(pdf[0].get_text()[:1000])

In [7]:
# ============================================================
# CELL 4.3 — Text Chunking Strategy
# ============================================================
# Chunking is one of the most important RAG parameters!
# - chunk_size: number of tokens per chunk
# - chunk_overlap: overlap between consecutive chunks (prevents context loss)

# Define chunking parameters
CHUNK_SIZE = 512        # tokens per chunk
CHUNK_OVERLAP = 64     # tokens of overlap

# Create the text splitter
text_splitter = SentenceSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
)

# Split documents into nodes (chunks)
nodes = text_splitter.get_nodes_from_documents(documents)

print(f"📊 Chunking Results:")
print(f"   Input documents: {len(documents)} pages")
print(f"   Output chunks:   {len(nodes)} nodes")
print(f"   Chunk size:      {CHUNK_SIZE} tokens")
print(f"   Chunk overlap:   {CHUNK_OVERLAP} tokens")

# Show chunk size distribution
chunk_lengths = [len(n.text.split()) for n in nodes]
print(f"\n📏 Chunk length statistics (in words):")
print(f"   Min:  {min(chunk_lengths)}")
print(f"   Max:  {max(chunk_lengths)}")
print(f"   Avg:  {sum(chunk_lengths) / len(chunk_lengths):.0f}")

# Display first 2 chunks
print("\n🔍 First 2 chunks preview:")
for i, node in enumerate(nodes[:2]):
    print(f"\n--- Chunk {i+1} ---")
    print(f"Source: {node.metadata.get('file_name', 'N/A')}")
    print(f"Text: {node.text[:300]}...")

📊 Chunking Results:
   Input documents: 3 pages
   Output chunks:   4 nodes
   Chunk size:      512 tokens
   Chunk overlap:   64 tokens

📏 Chunk length statistics (in words):
   Min:  42
   Max:  316
   Avg:  226

🔍 First 2 chunks preview:

--- Chunk 1 ---
Source: ai_research_report.pdf
Text: ## Artificial Intelligence Research Report 2024

## Section 1: Executive Summary

This report examines the state of artificial intelligence research in 2024, with a focus on large language models, multimodal systems, and real-world deployment challenges.

Key findings include: (1) LLM capabilities h...

--- Chunk 2 ---
Source: ai_research_report.pdf
Text: June 2024: The EU AI Act came into effect, requiring risk assessments for high-risk AI systems.

September 2024: Meta released LLaMA 3.1 with a 405B parameter variant under open license.

December 2024: The first international AI Safety Summit produced the Seoul Declaration....


---
## 🧩 Section 5 — Embeddings

> **What are embeddings?** Embeddings are numerical vector representations of text.
> Similar texts have similar vectors (measured by cosine similarity).
>
> We provide **two options**:
> - **Primary:** `nomic-embed-text` via Ollama (local, 768-dim vectors)
> - **Fallback:** `sentence-transformers/all-MiniLM-L6-v2` (auto-download, 384-dim)

In [ ]:
# ============================================================
# CELL 5.0 — Pull Ollama Embedding Model
# ============================================================
# Run this in your terminal (outside notebook):
#   ollama pull nomic-embed-text

# Or run here (will take a few minutes on first run):
if ollama_running:
    print("📥 Pulling nomic-embed-text embedding model...")
    result = subprocess.run(
        ["ollama", "pull", "nomic-embed-text"],
        capture_output=True, text=True, timeout=300
    )
    if result.returncode == 0:
        print("✅ nomic-embed-text ready!")
    else:
        print(f"⚠️  Could not pull: {result.stderr}")
else:
    print("⏭️  Ollama not running. Will use sentence-transformers fallback.")

In [9]:
# ============================================================
# CELL 5.1 — Setup Embedding Model
# ============================================================

embed_model = None
embed_model_name = None

# ----- PRIMARY: Ollama nomic-embed-text -----
if ollama_running:
    try:
        from llama_index.embeddings.ollama import OllamaEmbedding
        
        embed_model = OllamaEmbedding(
            model_name="nomic-embed-text",
            base_url="http://localhost:11434",
            embed_batch_size=10,  # Process 10 chunks at once
        )
        
        # Test it
        test_vec = embed_model.get_text_embedding("test embedding")
        embed_model_name = "nomic-embed-text (Ollama)"
        print(f"✅ Using Ollama: nomic-embed-text")
        print(f"   Embedding dimensions: {len(test_vec)}")
        
    except Exception as e:
        print(f"⚠️  Ollama embedding failed: {e}")
        embed_model = None

# ----- FALLBACK: sentence-transformers -----
if embed_model is None:
    from llama_index.embeddings.huggingface import HuggingFaceEmbedding
    os.environ["HF_TOKEN"] = os.environ.get("HF_TOKEN")
    print("⏭️  Using HuggingFace sentence-transformers as fallback embedding model.")
    print(os.environ.get("HF_TOKEN"))
    # Downloads ~90MB on first run
    embed_model = HuggingFaceEmbedding(
        model_name="sentence-transformers/all-MiniLM-L6-v2",
        embed_batch_size=32,
    )
    
    test_vec = embed_model.get_text_embedding("test embedding")
    embed_model_name = "all-MiniLM-L6-v2 (sentence-transformers)"
    print(f"✅ Using HuggingFace: all-MiniLM-L6-v2")
    print(f"   Embedding dimensions: {len(test_vec)}")

# Apply globally
Settings.embed_model = embed_model
print(f"\n🎯 Active embedding model: {embed_model_name}")

✅ Using Ollama: nomic-embed-text
   Embedding dimensions: 768

🎯 Active embedding model: nomic-embed-text (Ollama)


In [10]:
# ============================================================
# CELL 5.2 — Embedding Visualization Demo
# ============================================================
# Show how similar texts have similar vectors

import numpy as np

texts = [
    "Artificial intelligence and machine learning",
    "AI and deep learning systems",              # Similar to #1
    "Climate change and carbon emissions",        # Different topic
]

embeddings = [embed_model.get_text_embedding(t) for t in texts]

def cosine_sim(a, b):
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

print("🔬 Embedding Similarity Demo:")
print(f"\nText 1: '{texts[0]}'")
print(f"Text 2: '{texts[1]}'")
print(f"Text 3: '{texts[2]}'")
print()
print(f"Similarity (Text 1 vs Text 2 — similar topic): {cosine_sim(embeddings[0], embeddings[1]):.4f}")
print(f"Similarity (Text 1 vs Text 3 — different topic): {cosine_sim(embeddings[0], embeddings[2]):.4f}")
print("\n💡 Higher score = more semantically similar. Range: -1 to 1")

🔬 Embedding Similarity Demo:

Text 1: 'Artificial intelligence and machine learning'
Text 2: 'AI and deep learning systems'
Text 3: 'Climate change and carbon emissions'

Similarity (Text 1 vs Text 2 — similar topic): 0.8086
Similarity (Text 1 vs Text 3 — different topic): 0.4149

💡 Higher score = more semantically similar. Range: -1 to 1


---
## 🗄️ Section 6 — Build Vector Index with ChromaDB

> **ChromaDB** is an open-source vector database. It stores embeddings persistently on disk,
> so you don't need to re-embed documents every time you run the notebook.

> **💡 Trainer Note:** Show students the `.chroma` folder that appears on disk after this cell.

In [11]:
# ============================================================
# CELL 6.1 — Create ChromaDB + Build Vector Index
# ============================================================

import shutil

CHROMA_DB_PATH = "./chroma_db"        # Persistent storage path
CHROMA_COLLECTION = "rag_lab_docs"    # Collection name

# Optional: reset DB (uncomment to start fresh)
# if os.path.exists(CHROMA_DB_PATH):
#     shutil.rmtree(CHROMA_DB_PATH)
#     print("🗑️  Cleared existing ChromaDB")

# Initialize ChromaDB
chroma_client = chromadb.PersistentClient(path=CHROMA_DB_PATH)

# Get or create collection
chroma_collection = chroma_client.get_or_create_collection(
    name=CHROMA_COLLECTION,
    metadata={"hnsw:space": "cosine"}   # Use cosine similarity
)

# Wrap in LlamaIndex vector store
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

print("⚡ Building vector index (embedding all chunks)...")
print(f"   This embeds {len(nodes)} chunks — may take 1-3 minutes...")

# Build the index
index = VectorStoreIndex(
    nodes,
    storage_context=storage_context,
    show_progress=True,
)

print("\n✅ Vector index built and persisted!")
print(f"   DB path: {CHROMA_DB_PATH}")
print(f"   Collection: {CHROMA_COLLECTION}")
print(f"   Total vectors: {chroma_collection.count()}")

⚡ Building vector index (embedding all chunks)...
   This embeds 4 chunks — may take 1-3 minutes...


Generating embeddings: 100%|██████████| 4/4 [00:05<00:00,  1.37s/it]


✅ Vector index built and persisted!
   DB path: ./chroma_db
   Collection: rag_lab_docs
   Total vectors: 4


In [13]:
# ============================================================
# CELL 6.2 — Inspect Vector DB Contents
# ============================================================

# Peek at stored vectors
results = chroma_collection.peek(limit=3)

print("🔍 ChromaDB Inspection:")
print(f"   Total documents: {chroma_collection.count()}")
print(f"\n📋 Sample entries:")
for i, (doc_id, metadata, doc) in enumerate(zip(
    results['ids'], results['metadatas'], results['documents']
)):
    print(f"\n  Entry {i+1}:")
    print(f"    ID: {doc_id}")
    print(f"    Metadata: {metadata}")
    print(f"    Text preview: {doc[:150]}...")

# Show embedding dimensions
if results.get("embeddings") is not None and len(results["embeddings"]) > 0:
    print(f"\n📐 Embedding dimensions: {len(results['embeddings'][0])}")

print("\n✅ ChromaDB inspection complete")

🔍 ChromaDB Inspection:
   Total documents: 4

📋 Sample entries:

  Entry 1:
    ID: fd82e9e8-73da-4bd2-88f0-78082ce56063
    Metadata: {'ref_doc_id': 'acae30b0-d1fe-4b8c-acb0-0b1697fd2b62', '_node_content': '{"id_": "fd82e9e8-73da-4bd2-88f0-78082ce56063", "embedding": null, "metadata": {"file_name": "ai_research_report.pdf"}, "excluded_embed_metadata_keys": [], "excluded_llm_metadata_keys": [], "relationships": {"1": {"node_id": "acae30b0-d1fe-4b8c-acb0-0b1697fd2b62", "node_type": "4", "metadata": {"file_name": "ai_research_report.pdf"}, "hash": "e04676512cd0e944b508db004d2813bf6786603d2bc4d2fe0884c1c52e1312c7", "class_name": "RelatedNodeInfo"}, "3": {"node_id": "7060e4d5-0f1d-42a9-a404-c07b26aba7c2", "node_type": "1", "metadata": {}, "hash": "42ae680c871323199323a41c2acc4ca0b2d9896cedc6c0a9ab974fb4bba5d02e", "class_name": "RelatedNodeInfo"}}, "metadata_template": "{key}: {value}", "metadata_separator": "\\n", "text": "", "mimetype": "text/plain", "start_char_idx": 0, "end_char_idx": 2

In [14]:
# ============================================================
# CELL 6.3 — Load Index from Disk (skip re-embedding)
# ============================================================
# In production: load existing index instead of rebuilding
# Uncomment this block and comment out Cell 6.1 on subsequent runs

# chroma_client = chromadb.PersistentClient(path=CHROMA_DB_PATH)
# chroma_collection = chroma_client.get_collection(CHROMA_COLLECTION)
# vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
# storage_context = StorageContext.from_defaults(vector_store=vector_store)
# index = VectorStoreIndex.from_vector_store(
#     vector_store,
#     storage_context=storage_context,
# )
# print("✅ Index loaded from disk - no re-embedding needed!")

print("💡 TIP: On subsequent runs, uncomment Cell 6.3 to skip re-embedding")
print(f"   ChromaDB stores vectors at: {os.path.abspath(CHROMA_DB_PATH)}")

💡 TIP: On subsequent runs, uncomment Cell 6.3 to skip re-embedding
   ChromaDB stores vectors at: d:\Trainings\AI\abu dhabhi agent\extra lab\chroma_db


---
## 🔗 Section 7 — Basic RAG Pipeline

> This builds the full RAG pipeline:
> 1. **Retriever** — Finds relevant chunks from ChromaDB
> 2. **Response Synthesizer** — Combines chunks into a coherent answer
> 3. **Query Engine** — Ties everything together
>
> We also show the **retrieved source chunks** so you can inspect what the LLM actually sees.

In [25]:
# ============================================================
# CELL 7.1 — Build the RAG Query Engine (UPDATED)
# ============================================================

TOP_K = 4

retriever = VectorIndexRetriever(
    index=index,
    similarity_top_k=TOP_K,
)

# IMPORTANT FIX:
# Disable structured_answer_filtering to avoid JSON parsing errors
response_synthesizer = get_response_synthesizer(
    response_mode="compact",
    structured_answer_filtering=False,   # ← FIXED
)

query_engine = RetrieverQueryEngine(
    retriever=retriever,
    response_synthesizer=response_synthesizer,
)

print("✅ RAG Pipeline Components:")
print(f"   Retriever:    VectorIndexRetriever (top_k={TOP_K})")
print(f"   Synthesizer:  compact mode (no JSON filtering)")
print(f"   LLM:          {type(llm).__name__}")
print(f"   Embed model:  {embed_model_name}")
print(f"   Vector DB:    ChromaDB ({chroma_collection.count()} vectors)")


✅ RAG Pipeline Components:
   Retriever:    VectorIndexRetriever (top_k=4)
   Synthesizer:  compact mode (no JSON filtering)
   LLM:          OpenAILike
   Embed model:  nomic-embed-text (Ollama)
   Vector DB:    ChromaDB (4 vectors)


In [26]:
# ============================================================
# CELL 7.2 — RAG Query Helper Function (UPDATED)
# ============================================================

def rag_query(question: str, show_sources: bool = True) -> str:
    """
    Full RAG query: retrieve relevant chunks + generate answer.
    """

    print(f"\n{'='*65}")
    print(f"❓ QUESTION: {question}")
    print('='*65)

    # Execute RAG query
    response = query_engine.query(question)

    # Show retrieved source chunks
    if show_sources and hasattr(response, 'source_nodes'):
        print(f"\n🔍 Retrieved {len(response.source_nodes)} source chunks:")
        for i, node in enumerate(response.source_nodes):

            # SAFE SCORE FORMAT
            score = f"{node.score:.4f}" if node.score is not None else "N/A"

            print(f"\n  📄 Source {i+1} (score: {score}):")
            print(f"     File: {node.metadata.get('file_name', 'N/A')}")
            print(f"     Page: {node.metadata.get('page_label', 'N/A')}")
            preview = node.text[:200].replace('\n', ' ')
            print(f"     Text: {preview}...")

    # Show final answer
    print(f"\n💬 ANSWER:")
    print("-" * 65)

    answer = str(response)
    for line in textwrap.wrap(answer, width=65):
        print(line)

    print("-" * 65)

    return answer

print("✅ RAG query helper ready!")


✅ RAG query helper ready!


In [27]:
# ============================================================
# CELL 7.3 — Test Basic RAG Queries
# ============================================================

# Test Query 1: Summarization
answer1 = rag_query(
    "Summarize the key findings from Section 1 of the AI research report."
)


❓ QUESTION: Summarize the key findings from Section 1 of the AI research report.

🔍 Retrieved 4 source chunks:

  📄 Source 1 (score: 0.7746):
     File: ai_research_report.pdf
     Page: N/A
     Text: ## Artificial Intelligence Research Report 2024  ## Section 1: Executive Summary  This report examines the state of artificial intelligence research in 2024, with a focus on large language models, mul...

  📄 Source 2 (score: 0.7539):
     File: ai_research_report.pdf
     Page: N/A
     Text: June 2024: The EU AI Act came into effect, requiring risk assessments for high-risk AI systems.  September 2024: Meta released LLaMA 3.1 with a 405B parameter variant under open license.  December 202...

  📄 Source 3 (score: 0.6559):
     File: ml_technical_guide.pdf
     Page: N/A
     Text: ## Practical Machine Learning: A Technical Guide  ## Section 1: Introduction to ML Systems  Machine learning systems consist of three core components: data pipelines, model training infrastructure, an...

  

In [28]:
# Test Query 2: Comparison
answer2 = rag_query(
    "Compare the carbon emission trends between developed and developing nations."
)


❓ QUESTION: Compare the carbon emission trends between developed and developing nations.

🔍 Retrieved 4 source chunks:

  📄 Source 1 (score: 0.7444):
     File: climate_policy_brief.pdf
     Page: N/A
     Text: ## Climate Change Policy Brief: 2024 Assessment  ## Section 1: Overview  Global average temperatures in 2024 exceeded pre-industrial levels by 1.5°C for the first time in recorded history, crossing th...

  📄 Source 2 (score: 0.5921):
     File: ai_research_report.pdf
     Page: N/A
     Text: June 2024: The EU AI Act came into effect, requiring risk assessments for high-risk AI systems.  September 2024: Meta released LLaMA 3.1 with a 405B parameter variant under open license.  December 202...

  📄 Source 3 (score: 0.5741):
     File: ml_technical_guide.pdf
     Page: N/A
     Text: ## Practical Machine Learning: A Technical Guide  ## Section 1: Introduction to ML Systems  Machine learning systems consist of three core components: data pipelines, model training infrastructure,

In [29]:
# Test Query 3: Date/Event extraction
answer3 = rag_query(
    "Extract all dates and events mentioned in the documents."
)


❓ QUESTION: Extract all dates and events mentioned in the documents.

🔍 Retrieved 4 source chunks:

  📄 Source 1 (score: 0.6172):
     File: ai_research_report.pdf
     Page: N/A
     Text: June 2024: The EU AI Act came into effect, requiring risk assessments for high-risk AI systems.  September 2024: Meta released LLaMA 3.1 with a 405B parameter variant under open license.  December 202...

  📄 Source 2 (score: 0.5828):
     File: climate_policy_brief.pdf
     Page: N/A
     Text: ## Climate Change Policy Brief: 2024 Assessment  ## Section 1: Overview  Global average temperatures in 2024 exceeded pre-industrial levels by 1.5°C for the first time in recorded history, crossing th...

  📄 Source 3 (score: 0.5795):
     File: ai_research_report.pdf
     Page: N/A
     Text: ## Artificial Intelligence Research Report 2024  ## Section 1: Executive Summary  This report examines the state of artificial intelligence research in 2024, with a focus on large language models, mul...

  📄 Source 4

In [30]:
# Test Query 4: Specific concept
answer4 = rag_query(
    "What do the documents say about RAG systems and vector databases?"
)


❓ QUESTION: What do the documents say about RAG systems and vector databases?

🔍 Retrieved 4 source chunks:

  📄 Source 1 (score: 0.6429):
     File: ml_technical_guide.pdf
     Page: N/A
     Text: ## Practical Machine Learning: A Technical Guide  ## Section 1: Introduction to ML Systems  Machine learning systems consist of three core components: data pipelines, model training infrastructure, an...

  📄 Source 2 (score: 0.6245):
     File: ai_research_report.pdf
     Page: N/A
     Text: ## Artificial Intelligence Research Report 2024  ## Section 1: Executive Summary  This report examines the state of artificial intelligence research in 2024, with a focus on large language models, mul...

  📄 Source 3 (score: 0.6233):
     File: ai_research_report.pdf
     Page: N/A
     Text: June 2024: The EU AI Act came into effect, requiring risk assessments for high-risk AI systems.  September 2024: Meta released LLaMA 3.1 with a 405B parameter variant under open license.  December 202...

  📄 S

---
## 🤖 Section 8 — Agentic RAG

> **What is Agentic RAG?** Instead of a fixed pipeline, an *agent* decides **when** to retrieve,
> **how** to rewrite queries, and can perform **multi-hop** reasoning across multiple tool calls.
>
> ```
> User Query
>     ↓
> [AGENT] ← LLM with reasoning
>     ↓ decides
>  ┌──────────────────────────────┐
>  │  Tool 1: search_documents()  │ ← RAG retrieval
>  │  Tool 2: summarize_topic()   │ ← targeted summary
>  │  Tool 3: compare_topics()    │ ← comparison task
>  └──────────────────────────────┘
>     ↓ loops until done
>  Final Answer + Citations
> ```
>
> **Key Benefits:**
> - Multi-step reasoning
> - Query rewriting and decomposition
> - Adaptive retrieval strategy
> - Self-correction

In [31]:
# ============================================================
# CELL 8.1 — Create Separate Topic Indexes for Agent Tools
# ============================================================
# Each tool covers a specific document domain
# This gives the agent more precise retrieval control

from llama_index.core import SimpleDirectoryReader, VectorStoreIndex, Document

def create_topic_index(file_path: str, collection_name: str):
    """Create a vector index for a single document."""
    # Load single file
    reader = SimpleDirectoryReader(input_files=[file_path])
    docs = reader.load_data()
    
    # Create ChromaDB collection
    coll = chroma_client.get_or_create_collection(
        name=collection_name,
        metadata={"hnsw:space": "cosine"}
    )
    vs = ChromaVectorStore(chroma_collection=coll)
    sc = StorageContext.from_defaults(vector_store=vs)
    
    # Build index
    idx = VectorStoreIndex.from_documents(docs, storage_context=sc)
    return idx

print("🏗️  Building topic-specific indexes for agent tools...")

# Build separate indexes per topic
ai_index = create_topic_index("sample_docs/ai_research_report.pdf", "agent_ai_docs")
print("  ✅ AI Research index ready")

climate_index = create_topic_index("sample_docs/climate_policy_brief.pdf", "agent_climate_docs")
print("  ✅ Climate Policy index ready")

ml_index = create_topic_index("sample_docs/ml_technical_guide.pdf", "agent_ml_docs")
print("  ✅ ML Technical Guide index ready")

print("\n✅ All topic indexes built!")

🏗️  Building topic-specific indexes for agent tools...
  ✅ AI Research index ready
  ✅ Climate Policy index ready
  ✅ ML Technical Guide index ready

✅ All topic indexes built!


In [32]:
# ============================================================
# CELL 8.2 — Define Agent Tools
# ============================================================
# Each QueryEngineTool is a callable tool the agent can invoke.
# The description is critical — the agent uses it to decide WHEN to call the tool.

# Create query engines for each topic index
ai_engine = ai_index.as_query_engine(similarity_top_k=3)
climate_engine = climate_index.as_query_engine(similarity_top_k=3)
ml_engine = ml_index.as_query_engine(similarity_top_k=3)

# Wrap as agent tools
tools = [
    QueryEngineTool(
        query_engine=ai_engine,
        metadata=ToolMetadata(
            name="ai_research_tool",
            description=(
                "Use this tool to answer questions about artificial intelligence, "
                "large language models (LLMs), RAG systems, vector databases, "
                "AI market statistics, and AI events/milestones in 2024. "
                "Input should be a specific question about AI research."
            ),
        ),
    ),
    QueryEngineTool(
        query_engine=climate_engine,
        metadata=ToolMetadata(
            name="climate_policy_tool",
            description=(
                "Use this tool to answer questions about climate change, "
                "carbon emissions, renewable energy, climate policy, "
                "environmental regulations, and global temperature data. "
                "Input should be a specific question about climate or energy policy."
            ),
        ),
    ),
    QueryEngineTool(
        query_engine=ml_engine,
        metadata=ToolMetadata(
            name="ml_technical_tool",
            description=(
                "Use this tool to answer questions about machine learning techniques, "
                "model training, evaluation metrics, deployment strategies, MLOps, "
                "and data science best practices. "
                "Input should be a specific technical ML question."
            ),
        ),
    ),
]

print("✅ Agent tools created:")
for t in tools:
    print(f"   🔧 {t.metadata.name}")
    print(f"      {t.metadata.description[:80]}...")

✅ Agent tools created:
   🔧 ai_research_tool
      Use this tool to answer questions about artificial intelligence, large language ...
   🔧 climate_policy_tool
      Use this tool to answer questions about climate change, carbon emissions, renewa...
   🔧 ml_technical_tool
      Use this tool to answer questions about machine learning techniques, model train...


In [33]:
# ============================================================
# CELL 8.3 — Create ReAct Agent
# ============================================================
# ReAct = Reason + Act
# The agent loop:
#   1. THINK: reason about what to do
#   2. ACT: call a tool
#   3. OBSERVE: see the result
#   4. Repeat until answer is ready

# Custom system prompt for the agent
AGENT_SYSTEM_PROMPT = """
You are an expert research analyst with access to multiple knowledge bases.
Your job is to answer questions by:
1. Breaking down complex questions into sub-questions
2. Using the appropriate tool for each sub-question
3. Synthesizing information from multiple sources
4. Providing well-structured answers with citations
5. Being explicit about which document/tool provided each piece of information

Always cite your sources. If a tool doesn't have the answer, say so clearly.
For comparison questions, query multiple tools and synthesize the results.
"""

agent = ReActAgent.from_tools(
    tools=tools,
    llm=llm,
    verbose=True,             # Shows the reasoning trace!
    max_iterations=8,         # Max tool calls per query
    context=AGENT_SYSTEM_PROMPT,
)

print("✅ ReAct Agent created!")
print(f"   Max iterations: 8")
print(f"   Tools available: {[t.metadata.name for t in tools]}")
print(f"   LLM: {type(llm).__name__}")
print("\n💡 TIP: verbose=True shows the agent's reasoning trace (Thought → Action → Observation)")

✅ ReAct Agent created!
   Max iterations: 8
   Tools available: ['ai_research_tool', 'climate_policy_tool', 'ml_technical_tool']
   LLM: OpenAILike

💡 TIP: verbose=True shows the agent's reasoning trace (Thought → Action → Observation)


d:\Trainings\AI\abu dhabhi agent\.venv\Lib\site-packages\llama_index\core\agent\react\base.py:154: DeprecationWarning: Call to deprecated class ReActAgent. (ReActAgent has been rewritten and replaced by llama_index.core.agent.workflow.ReActAgent.

This implementation will be removed in a v0.13.0 and the new implementation will be promoted to the `from llama_index.core.agent import ReActAgent` path.

See the docs for more information: https://docs.llamaindex.ai/en/stable/understanding/agent/)
  return cls(
d:\Trainings\AI\abu dhabhi agent\.venv\Lib\site-packages\deprecated\classic.py:184: DeprecationWarning: Call to deprecated class AgentRunner. (AgentRunner has been deprecated and is not maintained.

This implementation will be removed in a v0.13.0.

See the docs for more information on updated agent usage: https://docs.llamaindex.ai/en/stable/understanding/agent/)
  return old_new1(cls, *args, **kwargs)


In [34]:
# ============================================================
# CELL 8.4 — Agentic RAG Query Helper
# ============================================================

def agent_query(question: str) -> str:
    """
    Run a query through the Agentic RAG pipeline.
    The agent will reason, select tools, and produce a cited answer.
    """
    print(f"\n{'🤖 ' + '='*60}")
    print(f"  AGENTIC RAG QUERY")
    print(f"{'='*62}")
    print(f"❓ Question: {question}")
    print(f"{'='*62}")
    print("\n🔄 Agent Reasoning Trace:")
    print("-" * 62)
    
    response = agent.chat(question)
    
    print("-" * 62)
    print("\n🎯 FINAL ANSWER:")
    print("=" * 62)
    answer = str(response)
    for line in textwrap.wrap(answer, width=65):
        print(line)
    print("=" * 62)
    return answer

print("✅ Agentic query helper ready!")

✅ Agentic query helper ready!


---
## 🎭 Section 9 — End-to-End Scenario

> **Scenario:** You are a research analyst. A client has asked for a comprehensive briefing
> using the three documents in the knowledge base. You will use the Agentic RAG system
> to answer complex, multi-document questions with full citations.
>
> **Notice:** Watch how the agent's reasoning trace shows:
> - Which tool it selects and why
> - How it combines information from multiple sources
> - How it produces a structured, cited answer

In [35]:
# ============================================================
# CELL 9.1 — Scenario Query 1: Multi-Document Summarization
# ============================================================

result1 = agent_query(
    "Summarize the key findings from the AI research report, "
    "focusing on what it says about Section 2 on large language models."
)


🤖 ============================================================
  AGENTIC RAG QUERY
❓ Question: Summarize the key findings from the AI research report, focusing on what it says about Section 2 on large language models.

🔄 Agent Reasoning Trace:
--------------------------------------------------------------
> Running step 5e47ea2b-7f15-4fd6-bdcd-c93068a59f8c. Step input: Summarize the key findings from the AI research report, focusing on what it says about Section 2 on large language models.
Thought: To summarize the key findings from the AI research report focusing on Section 2 about large language models, I need to use the ai_research_tool to get detailed information about this section.
Action: ai_research_tool
Action Input: {'input': 'Summarize the key findings from the AI research report, focusing on what it says about Section 2 on large language models.'}
Observation: The provided context does not contain the actual content of the AI research report, including any specific details 

In [36]:
# ============================================================
# CELL 9.2 — Scenario Query 2: Cross-Document Comparison
# ============================================================

result2 = agent_query(
    "Compare the growth of renewable energy technology (from the climate report) "
    "with the growth of AI technology (from the AI research report). "
    "What are the similarities in their adoption curves?"
)


🤖 ============================================================
  AGENTIC RAG QUERY
❓ Question: Compare the growth of renewable energy technology (from the climate report) with the growth of AI technology (from the AI research report). What are the similarities in their adoption curves?

🔄 Agent Reasoning Trace:
--------------------------------------------------------------
> Running step f8afcb50-fa63-4494-b872-a7fd94080ea8. Step input: Compare the growth of renewable energy technology (from the climate report) with the growth of AI technology (from the AI research report). What are the similarities in their adoption curves?
Thought: To compare the growth of renewable energy technology with the growth of AI technology, I need to gather information on both technologies' adoption curves. I will use the `climate_policy_tool` to get insights on renewable energy technology and the `ai_research_tool` to get insights on AI technology.
Action: climate_policy_tool
Action Input: {'input': 'What

In [ ]:
# ============================================================
# CELL 9.3 — Scenario Query 3: Date & Event Extraction
# ============================================================

result3 = agent_query(
    "Extract all specific dates and events mentioned across all documents. "
    "Present them in chronological order with the source document for each."
)

In [ ]:
# ============================================================
# CELL 9.4 — Scenario Query 4: Technical Deep-Dive
# ============================================================

result4 = agent_query(
    "What do the documents say about evaluation metrics? "
    "Cover both ML model evaluation and RAG system evaluation."
)

In [ ]:
# ============================================================
# CELL 9.5 — Scenario Query 5: Policy & Recommendation Synthesis
# ============================================================

result5 = agent_query(
    "What recommendations or best practices are given across all the documents? "
    "Group them by domain (AI, climate, ML engineering)."
)


# Langgraph

### LangGraph Mini RAG Workflow

This cell documents a tiny RAG pipeline implemented as a `langgraph` state graph, not a document ingestion step.

Key points:

- `RAGState` defines the workflow state:
    - `question`
    - `rewritten_query`
    - `retrieved_docs`
    - `answer`

- The graph has three nodes:
    1. `rewrite_query(state)` — rewrites the original user question via `llm.complete(...)` to improve retrieval quality
    2. `retrieve_docs(state)` — uses the existing `retriever` object to fetch relevant chunks for the rewritten query
    3. `generate_answer(state)` — uses the top 4 retrieved docs as context and asks the LLM to answer the original question

- The flow is linear:
    - `rewrite` → `retrieve` → `generate`

- `workflow.compile()` turns the graph into an executable app
- `app.invoke(...)` runs the pipeline and returns the final state

Important clarification:

- This code does not add or index documents
- It uses the already-built index and retriever
- It simply wires the three-step RAG process into an explicit, stateful workflow

Use this cell to show how a RAG pipeline can be modeled as a sequence of stateful transformation steps rather than a single monolithic function.

In [39]:
from langgraph.graph import StateGraph
from typing import TypedDict, List

# Install langgraph if not already installed
!pip install -q langgraph


class RAGState(TypedDict):
    question: str
    rewritten_query: str
    retrieved_docs: List[str]
    answer: str

def rewrite_query(state: RAGState) -> RAGState:
    prompt = (
        "Rewrite the user query to improve retrieval quality. "
        "Keep the meaning the same and add useful keywords.\n\n"
        f"User query: {state['question']}"
    )
    response = llm.complete(prompt)
    rewritten = getattr(response, "text", str(response)).strip()
    return {**state, "rewritten_query": rewritten}

def retrieve_docs(state: RAGState) -> RAGState:
    nodes = retriever.retrieve(state["rewritten_query"])
    texts = [getattr(n, "text", str(n)) for n in nodes]
    return {**state, "retrieved_docs": texts}

def generate_answer(state: RAGState) -> RAGState:
    context = "\n\n".join(state["retrieved_docs"][:4])
    prompt = (
        "Use the context below to answer the question accurately.\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {state['question']}\n\n"
        "Answer:"
    )
    response = llm.complete(prompt)
    answer = getattr(response, "text", str(response)).strip()
    return {**state, "answer": answer}

workflow = StateGraph(RAGState)
workflow.add_node("rewrite", rewrite_query)
workflow.add_node("retrieve", retrieve_docs)
workflow.add_node("generate", generate_answer)

workflow.set_entry_point("rewrite")
workflow.add_edge("rewrite", "retrieve")
workflow.add_edge("retrieve", "generate")

app = workflow.compile()

result = app.invoke({
    "question": "Explain how RAG systems use vector databases and why they are important.",
    "rewritten_query": "",
    "retrieved_docs": [],
    "answer": "",
})

print("Rewritten query:", result["rewritten_query"])
print("\nRetrieved docs count:", len(result["retrieved_docs"]))
print("\nGenerated answer:")
print(result["answer"])

Rewritten query: Revised query: Explain how Retrieval-Augmented Generation (RAG) systems utilize vector databases and highlight why these databases are crucial for their functionality.

Retrieved docs count: 4

Generated answer:
RAG (Retrieval-Augmented Generation) systems use vector databases to store and retrieve information efficiently. These databases are crucial because they enable the system to quickly find relevant documents or passages that can be used to generate accurate and contextually appropriate responses.

Here’s a detailed explanation:

1. **Efficient Retrieval**: Vector databases store embeddings (numerical representations) of text data. When a query is made, the database retrieves the closest matches based on similarity scores calculated using these embeddings. This process is much faster and more efficient than traditional keyword-based searches, especially when dealing with large volumes of text.

2. **Hybrid Search**: Modern RAG systems often employ hybrid search t

---
## 🎓 Section 10 — Classroom Exercises

> **Instructions for Students:**
> Complete the exercises below. Each one teaches a key RAG engineering concept.
> Answers at the bottom of each exercise cell (hidden — try first!).

---

In [ ]:
# ============================================================
# 🏋️ EXERCISE 1 — Modify Chunk Size
# ============================================================
# TASK: Re-build the index with a SMALLER chunk size (256 tokens).
#       Query it and compare the results with chunk_size=512.
#       Which gives better answers? Why?
#
# HINT: Smaller chunks = more precise retrieval but less context
#        Larger chunks = more context but may retrieve irrelevant content

# ---- YOUR CODE HERE ----
# Step 1: Create new text splitter with chunk_size=256
# exercise_splitter = SentenceSplitter(chunk_size=???, chunk_overlap=???)

# Step 2: Create new nodes
# exercise_nodes = exercise_splitter.get_nodes_from_documents(documents)

# Step 3: Create new ChromaDB collection
# exercise_collection = chroma_client.get_or_create_collection("exercise_1_small_chunks")

# Step 4: Build new index
# exercise_index = VectorStoreIndex(...)

# Step 5: Query and compare
# exercise_engine = exercise_index.as_query_engine()
# result = exercise_engine.query("What are the key AI milestones?")
# print(result)

print("📝 Exercise 1: Modify chunk size")
print("   Complete the steps above!")
print(f"   Current chunk_size: {CHUNK_SIZE}")
print("   Target chunk_size: 256")

In [ ]:
# ============================================================
# 🏋️ EXERCISE 2 — Add a New PDF
# ============================================================
# TASK: Download a real PDF (Wikipedia article exported as PDF,
#       or any public domain document) and add it to the RAG system.
#
# HINT: Create a new topic index + tool, then add it to the agent's tool list

# Example: Download a PDF
import urllib.request

# Wikipedia PDF (public domain)
PDF_URL = "https://en.m.wikipedia.org/api/rest_v1/page/pdf/Artificial_intelligence"
PDF_PATH = "sample_docs/wikipedia_ai.pdf"

# ---- YOUR CODE HERE ----
# Step 1: Download the PDF
# urllib.request.urlretrieve(PDF_URL, PDF_PATH)

# Step 2: Create index from new PDF
# new_index = create_topic_index(PDF_PATH, "wiki_ai_docs")

# Step 3: Create new tool
# new_tool = QueryEngineTool(...)

# Step 4: Create new agent with extended tool list
# extended_tools = tools + [new_tool]
# extended_agent = ReActAgent.from_tools(extended_tools, llm=llm, verbose=True)

# Step 5: Test the new agent
# extended_agent.chat("What is machine learning according to Wikipedia?")

print("📝 Exercise 2: Add a new PDF")
print("   Complete the steps above!")
print("   Goal: Add a 4th knowledge source to the agent")

In [ ]:
# ============================================================
# 🏋️ EXERCISE 3 — Add Hybrid Search (BM25 + Vector)
# ============================================================
# TASK: Add BM25 keyword search alongside vector search.
#       Hybrid search combines exact keyword matching with semantic similarity.
#       Great for technical terms, proper nouns, and exact phrases.

!pip install -q llama-index-retrievers-bm25

from llama_index.retrievers.bm25 import BM25Retriever
from llama_index.core.retrievers import QueryFusionRetriever

# ---- YOUR CODE HERE ----
# Step 1: Create BM25 retriever from nodes
# bm25_retriever = BM25Retriever.from_defaults(nodes=nodes, similarity_top_k=3)

# Step 2: Create vector retriever
# vector_retriever = VectorIndexRetriever(index=index, similarity_top_k=3)

# Step 3: Fuse both retrievers
# hybrid_retriever = QueryFusionRetriever(
#     [vector_retriever, bm25_retriever],
#     similarity_top_k=4,
#     num_queries=1,   # Don't rewrite query
#     mode="reciprocal_rerank",  # Fusion strategy
# )

# Step 4: Build hybrid query engine
# hybrid_engine = RetrieverQueryEngine(retriever=hybrid_retriever)
# result = hybrid_engine.query("What is the RAGAS framework?")
# print(result)

print("📝 Exercise 3: Add hybrid search")
print("   BM25 = keyword search (exact terms)")
print("   Vector = semantic search (meaning)")
print("   Hybrid = best of both worlds!")

In [ ]:
# ============================================================
# 🏋️ EXERCISE 4 — Add Reranking
# ============================================================
# TASK: Add a cross-encoder reranker to improve retrieval quality.
#       Reranking re-scores retrieved chunks using a more powerful model.
#       It's slower but much more accurate.

!pip install -q llama-index-postprocessor-flag-embedding-reranker

# ---- YOUR CODE HERE ----
# from llama_index.postprocessor.flag_embedding_reranker import FlagEmbeddingReranker

# Step 1: Create reranker (uses BAAI/bge-reranker-base — free, ~450MB)
# reranker = FlagEmbeddingReranker(
#     model="BAAI/bge-reranker-base",
#     top_n=3,   # Keep top 3 after reranking
# )

# Step 2: Add to query engine as postprocessor
# reranked_engine = RetrieverQueryEngine(
#     retriever=retriever,
#     node_postprocessors=[reranker],
# )

# Step 3: Compare with and without reranking
# query = "What are the best practices for ML model deployment?"
# print("WITHOUT reranking:", query_engine.query(query))
# print("WITH reranking:", reranked_engine.query(query))

print("📝 Exercise 4: Add reranking")
print("   First retrieve: top 10 candidates")
print("   Then rerank: pick best 3")
print("   Result: much better precision!")

In [ ]:
# ============================================================
# 🏋️ EXERCISE 5 — Add Conversation Memory
# ============================================================
# TASK: Create a RAG chatbot that remembers conversation history.
#       Users should be able to ask follow-up questions.

from llama_index.core.memory import ChatMemoryBuffer
from llama_index.core.chat_engine import CondensePlusContextChatEngine

# ---- YOUR CODE HERE ----
# Step 1: Create chat memory
# memory = ChatMemoryBuffer.from_defaults(token_limit=4096)

# Step 2: Create chat engine with memory
# chat_engine = CondensePlusContextChatEngine.from_defaults(
#     retriever=retriever,
#     memory=memory,
#     llm=llm,
#     verbose=True,
# )

# Step 3: Multi-turn conversation
# response1 = chat_engine.chat("What does the AI report say about LLMs?")
# print("Turn 1:", response1)
# response2 = chat_engine.chat("Can you elaborate on the first point you mentioned?")
# print("Turn 2:", response2)  # Should remember context!
# response3 = chat_engine.chat("How does this compare to what the climate report says?")
# print("Turn 3:", response3)  # Cross-document follow-up

print("📝 Exercise 5: Add conversation memory")
print("   Try: 'What is RAG?' then 'Tell me more about point 2'")
print("   The system should remember the first answer!")

In [ ]:
# ============================================================
# 🏋️ EXERCISE 6 — Query Rewriting
# ============================================================
# TASK: Implement query rewriting to improve retrieval.
#       Query rewriting expands/rephrase the user query to increase recall.

from llama_index.core.indices.query.query_transform.base import HyDEQueryTransform
from llama_index.core.query_engine import TransformQueryEngine

# HyDE = Hypothetical Document Embeddings
# It generates a hypothetical answer and embeds THAT for retrieval
# Often dramatically improves retrieval quality!

# ---- YOUR CODE HERE ----
# Step 1: Create HyDE transform
# hyde = HyDEQueryTransform(include_original=True)

# Step 2: Wrap the query engine
# hyde_engine = TransformQueryEngine(query_engine, query_transform=hyde)

# Step 3: Compare results
# query = "deployment strategies"
# print("Regular RAG:", query_engine.query(query))
# print("HyDE RAG:", hyde_engine.query(query))

print("📝 Exercise 6: Query rewriting with HyDE")
print("   HyDE: Generate hypothetical answer → embed it → retrieve")
print("   Especially good for short or vague queries!")

In [ ]:
# ============================================================
# 🏋️ EXERCISE 7 — RAG Evaluation with RAGAS
# ============================================================
# TASK: Evaluate your RAG system using the RAGAS framework.
#       Measure: faithfulness, answer_relevancy, context_precision

!pip install -q ragas datasets

# from ragas import evaluate
# from ragas.metrics import faithfulness, answer_relevancy, context_precision
# from datasets import Dataset

# ---- YOUR CODE HERE ----
# Step 1: Create evaluation dataset
# eval_questions = [
#     "What is RAG?",
#     "What are the carbon emission trends?",
#     "What is the RAGAS framework?",
# ]

# Step 2: Run RAG and collect results
# eval_data = {"question": [], "answer": [], "contexts": [], "ground_truth": []}
# for q in eval_questions:
#     response = query_engine.query(q)
#     eval_data["question"].append(q)
#     eval_data["answer"].append(str(response))
#     eval_data["contexts"].append([n.text for n in response.source_nodes])
#     eval_data["ground_truth"].append("")  # Optional

# Step 3: Run RAGAS evaluation
# dataset = Dataset.from_dict(eval_data)
# result = evaluate(dataset, metrics=[faithfulness, answer_relevancy])
# print(result)

print("📝 Exercise 7: RAG Evaluation")
print("   Faithfulness: Is answer grounded in retrieved docs?")
print("   Answer Relevancy: Does answer address the question?")
print("   Context Precision: Are retrieved chunks relevant?")

In [ ]:
# ============================================================
# 🏋️ EXERCISE 8 — Change Embedding Model
# ============================================================
# TASK: Try a different embedding model and compare retrieval quality.
#       Compare: all-MiniLM-L6-v2 (384-dim) vs all-mpnet-base-v2 (768-dim)

from llama_index.embeddings.huggingface import HuggingFaceEmbedding

# Available free models (from HuggingFace):
# - "sentence-transformers/all-MiniLM-L6-v2"    # Fast, 384-dim, ~80MB
# - "sentence-transformers/all-mpnet-base-v2"   # Better quality, 768-dim, ~420MB
# - "BAAI/bge-small-en-v1.5"                    # Great quality, 384-dim, ~130MB
# - "BAAI/bge-large-en-v1.5"                    # Best quality, 1024-dim, ~1.3GB

# ---- YOUR CODE HERE ----
# new_embed = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")

# Rebuild index with new embedding model
# Settings.embed_model = new_embed
# new_collection = chroma_client.get_or_create_collection("bge_small_index")
# new_vs = ChromaVectorStore(chroma_collection=new_collection)
# new_sc = StorageContext.from_defaults(vector_store=new_vs)
# new_index = VectorStoreIndex(nodes, storage_context=new_sc)
# new_engine = new_index.as_query_engine()

# Compare retrieval quality
# query = "What are the evaluation metrics for RAG?"
# print("Old model:", query_engine.query(query))
# print("New model:", new_engine.query(query))

print("📝 Exercise 8: Change embedding model")
print("   Try BAAI/bge-small-en-v1.5 — better quality than MiniLM!")
print("   Remember: index must be rebuilt when changing embedding model")

---
## 🚀 Section 11 — Optional Extensions

> These are extension tasks for advanced students or self-study after the lab.
> Code stubs provided to guide implementation.

---

In [ ]:
# ============================================================
# 🚀 EXTENSION 1 — Streamlit UI
# ============================================================
# Create a web UI for your RAG system using Streamlit.
# Run with: streamlit run rag_app.py

streamlit_app_code = '''
# rag_app.py — Save this file and run: streamlit run rag_app.py

import streamlit as st
import os

st.set_page_config(page_title="📚 RAG Assistant", layout="wide")
st.title("📚 RAG Document Assistant")
st.markdown("Powered by LlamaIndex + ChromaDB + Ollama")

# Sidebar config
with st.sidebar:
    st.header("⚙️ Settings")
    use_agent = st.checkbox("Use Agentic RAG", value=True)
    top_k = st.slider("Top-K chunks", 1, 10, 4)
    show_sources = st.checkbox("Show sources", value=True)

# --- Initialize RAG (paste your setup code here) ---
# @st.cache_resource
# def load_rag_system():
#     ... (your index, agent, query engine setup)
#     return query_engine, agent

# Chat interface
if "messages" not in st.session_state:
    st.session_state.messages = []

for message in st.session_state.messages:
    with st.chat_message(message["role"]):
        st.markdown(message["content"])

if prompt := st.chat_input("Ask a question about your documents..."):
    st.session_state.messages.append({"role": "user", "content": prompt})
    with st.chat_message("user"):
        st.markdown(prompt)
    
    with st.chat_message("assistant"):
        with st.spinner("Searching documents..."):
            # response = agent.chat(prompt) if use_agent else query_engine.query(prompt)
            response = f"[Configure your RAG system here] Answer to: {prompt}"
            st.markdown(response)
    
    st.session_state.messages.append({"role": "assistant", "content": str(response)})
'''

with open("rag_app.py", "w") as f:
    f.write(streamlit_app_code)

print("✅ Streamlit app template created: rag_app.py")
print("   To run: pip install streamlit && streamlit run rag_app.py")

In [ ]:
# ============================================================
# 🚀 EXTENSION 2 — LangGraph Workflow Agent
# ============================================================
# LangGraph gives you explicit control over agent state and flow.
# Great for complex multi-step workflows.

# !pip install -q langgraph langchain-community langchain-core

langgraph_example = '''
from langgraph.graph import StateGraph, END
from typing import TypedDict, List

class RAGState(TypedDict):
    question: str
    rewritten_query: str
    retrieved_docs: List[str]
    answer: str
    iterations: int

# Node 1: Query Rewriting
def rewrite_query(state: RAGState) -> RAGState:
    rewritten = llm.complete(f"Rewrite this query for better retrieval: {state['question']}")
    return {**state, "rewritten_query": str(rewritten)}

# Node 2: Retrieve Documents
def retrieve(state: RAGState) -> RAGState:
    nodes = retriever.retrieve(state["rewritten_query"])
    docs = [n.text for n in nodes]
    return {**state, "retrieved_docs": docs}

# Node 3: Generate Answer
def generate(state: RAGState) -> RAGState:
    context = "\\n".join(state["retrieved_docs"])
    prompt = f"Context: {context}\\n\\nQuestion: {state['question']}\\n\\nAnswer:"
    answer = llm.complete(prompt)
    return {**state, "answer": str(answer)}

# Node 4: Grade relevance (decide if answer is good enough)
def grade_answer(state: RAGState) -> str:
    if state["iterations"] >= 2:
        return "done"
    grade = llm.complete(f"Is this answer complete? YES or NO.\\nAnswer: {state['answer']}")
    return "done" if "YES" in str(grade).upper() else "retry"

# Build the graph
workflow = StateGraph(RAGState)
workflow.add_node("rewrite", rewrite_query)
workflow.add_node("retrieve", retrieve)
workflow.add_node("generate", generate)

workflow.set_entry_point("rewrite")
workflow.add_edge("rewrite", "retrieve")
workflow.add_edge("retrieve", "generate")
workflow.add_conditional_edges("generate", grade_answer, {"done": END, "retry": "rewrite"})

app = workflow.compile()
result = app.invoke({"question": "What is RAG?", "iterations": 0})
print(result["answer"])
'''

print("📝 LangGraph Workflow Agent Pattern:")
print("   rewrite_query → retrieve → generate → grade → (retry or done)")
print("\nCode template:")
print(langgraph_example)

In [ ]:
# ============================================================
# 🐛 DEBUGGING TIPS & COMMON ISSUES
# ============================================================

debug_tips = {
    "Ollama not responding": [
        "Run 'ollama serve' in a separate terminal",
        "Check port: curl http://localhost:11434/api/tags",
        "On Mac: Check Activity Monitor for Ollama process"
    ],
    "Out of memory (OOM) errors": [
        "Use a smaller model: phi3:mini or gemma:2b",
        "Reduce batch_size in embedding model",
        "Reduce chunk_size to 256",
        "Process documents in smaller batches"
    ],
    "OpenRouter 401 error": [
        "Check API key starts with 'sk-or-v1-'",
        "Verify key at: https://openrouter.ai/keys",
        "Check if model is free: https://openrouter.ai/models"
    ],
    "Poor retrieval quality": [
        "Try larger chunk_size (1024) for complex docs",
        "Increase TOP_K to 8-10",
        "Use a better embedding model (bge-large)",
        "Add reranking (Exercise 4)",
        "Try hybrid search (Exercise 3)"
    ],
    "Agent stuck in loop": [
        "Reduce max_iterations to 5",
        "Improve tool descriptions to be more specific",
        "Add explicit stop conditions in system prompt"
    ],
    "ChromaDB dimension mismatch": [
        "Delete ./chroma_db folder and re-run",
        "Different embed models produce different dimensions",
        "Use different collection names for different models"
    ]
}

print("🐛 DEBUGGING GUIDE")
print("=" * 60)
for issue, solutions in debug_tips.items():
    print(f"\n❌ Issue: {issue}")
    for i, sol in enumerate(solutions, 1):
        print(f"   {i}. {sol}")

In [ ]:
# ============================================================
# 🎓 LAB SUMMARY
# ============================================================

print("""
╔══════════════════════════════════════════════════════════╗
║            🎓 RAG CLASSROOM LAB — SUMMARY               ║
╠══════════════════════════════════════════════════════════╣
║                                                          ║
║  ✅ What You Built:                                      ║
║     • PDF loading + chunking pipeline                    ║
║     • Local embeddings (Ollama + sentence-transformers)  ║
║     • ChromaDB persistent vector store                   ║
║     • Basic RAG query engine                             ║
║     • Agentic RAG with ReAct agent                       ║
║     • Multi-document, multi-tool agent                   ║
║     • End-to-end analyst scenario                        ║
║                                                          ║
║  🔧 Key Tools:                                           ║
║     • LlamaIndex — RAG orchestration                     ║
║     • ChromaDB — Vector storage                          ║
║     • Ollama — Local LLM + embeddings                    ║
║     • OpenRouter — Cloud LLM (free tier)                 ║
║     • nomic-embed-text — Local embeddings                ║
║                                                          ║
║  📚 Key Concepts:                                        ║
║     • Chunking strategy affects retrieval quality        ║
║     • Embedding dimensions must match in vector store    ║
║     • ReAct agents loop: Think → Act → Observe           ║
║     • Tool descriptions guide agent decisions            ║
║     • Hybrid search > pure vector search                 ║
║     • Reranking improves precision significantly         ║
║                                                          ║
║                               ║
╚══════════════════════════════════════════════════════════╝
""")